# 1. RAG Básico - Implementación con Streamlit

## Objetivos de Aprendizaje
- Comprender los componentes fundamentales de un sistema RAG
- Implementar una búsqueda por palabras clave simple (con filtro de stopwords)
- Generar respuestas basadas en contexto recuperado
- Construir una interfaz interactiva con Streamlit (sección final)

## ¿Qué es RAG (Retrieval-Augmented Generation)?

RAG es una arquitectura que combina dos fases para generar respuestas:
1. **Retrieval (Recuperación)**: Busca y recupera información relevante de una base de conocimiento.
2. **Generation (Generación)**: Utiliza la información recuperada como contexto para que un LLM genere una respuesta precisa y fundamentada.

### Ventajas:
- **Reduce alucinaciones**: Las respuestas se basan en datos concretos.
- **Conocimiento actualizado**: La base de conocimiento puede actualizarse fácilmente.
- **Transparencia**: Se puede citar la fuente de la información.

### Componentes de nuestro RAG Básico:
- **Base de documentos**: Una lista de strings en memoria (en producción sería una base vectorial).
- **Mecanismo de retrieval**: Búsqueda simple por coincidencia de palabras clave (con filtro de stopwords).
- **Generador**: Un LLM que recibe el contexto y la pregunta.

## Configuración Inicial

Este notebook usa **OpenRouter** con un modelo gratuito. Si no tienes clave, el notebook te la pedirá interactivamente.

In [ ]:
# Instalación de dependencias (celda de código, no markdown)
!pip install openai python-dotenv streamlit -q
print("✅ Dependencias instaladas")

In [ ]:
# Importaciones y configuración
import os
from openai import OpenAI
from dotenv import load_dotenv
from getpass import getpass
import json

# Cargar variables desde .env (si existe)
load_dotenv()

print("✅ Librerías importadas correctamente")

In [ ]:
# Inicialización del cliente OpenAI (apuntando a OpenRouter)
BASE_URL = "https://openrouter.ai/api/v1"
API_KEY = os.environ.get("OPENROUTER_API_KEY")

if not API_KEY:
    print("🔑 No se encontró OPENROUTER_API_KEY en el entorno.")
    API_KEY = getpass("Ingresa tu clave de OpenRouter (sk-or-v1-...): ")
    os.environ["OPENROUTER_API_KEY"] = API_KEY

if not API_KEY or len(API_KEY) < 20:
    raise ValueError("❌ Clave inválida. Debe empezar por 'sk-or-v1-...'")

# Cliente
client = OpenAI(base_url=BASE_URL, api_key=API_KEY)

# Headers opcionales para identificar tu app
DEFAULT_HEADERS = {
    "HTTP-Referer": "http://localhost:8000",
    "X-Title": "RAG Básico",
}

# Modelo por defecto (gratuito)
MODELO = "minimax/minimax-m3:free"

print("✅ Cliente OpenRouter inicializado")
print(f"   Modelo por defecto: {MODELO}")

# Base de documentos

Cargamos la base de documentos que vamos a utilizar para el RAG. En este caso, contiene información sobre **inteligencia artificial y LLMs**.

> 💡 **Nota pedagógica**: En un sistema RAG real, esta base vendría de una base de datos vectorial (como FAISS, Chroma o Pinecone) con embeddings. Aquí usamos una lista simple para fines didácticos.

In [ ]:
documents = [
    "La inteligencia artificial es una rama de la informática que busca crear máquinas capaces de realizar tareas que requieren inteligencia humana.",
    "Los modelos de lenguaje grande (LLM) son sistemas de IA entrenados en enormes cantidades de texto para generar y comprender lenguaje natural.",
    "RAG (Retrieval-Augmented Generation) combina la búsqueda de información relevante con la generación de texto para producir respuestas más precisas.",
    "LangChain es un framework que facilita el desarrollo de aplicaciones con modelos de lenguaje, proporcionando herramientas para cadenas y agentes.",
    "El prompt engineering es la práctica de diseñar instrucciones efectivas para obtener los mejores resultados de los modelos de IA."
]

print(f"📚 Base de documentos creada con {len(documents)} documentos:")
for i, doc in enumerate(documents, 1):
    print(f"{i}. {doc[:60]}...")

# Función de recuperación simple (con filtro de stopwords)

La función `simple_retrieval()` busca documentos que contengan palabras clave de la consulta. **Importante**: filtramos las stopwords (palabras vacías como "es", "de", "qué", "cómo") porque de lo contrario cualquier pregunta coincidiría con todos los documentos.

### Stopwords en español más comunes:
`de, la, que, el, en, y, a, los, se, del, las, un, por, con, no, una, su, para, es, al, lo, como, más, o, pero, sus, le, ha, me, si, sin, sobre, este, ya, entre, cuando, todo, esta, ser, son, dos, también, fue, había, era, muy, años, hasta, desde, está, mi, porque, qué, sólo, han, yo, hay, vez, puede, todos, así, nos, ni, parte, tiene, él, uno, donde, bien, tiempo, mismo, ese, ahora, cada, e, vida, otro, después, te, otros, aunque, esa, eso, hace, otra, gobierno, tan, durante, siempre, día, vez, grande, cosas, política, pequeño, embargo, puesto, tipo, ahí, aquel, sea, según, menos, mundo, fue, año, dijo, sido, tiene, hacer, mientras, cual, cómo`

In [ ]:
# Lista de stopwords en español (versión simplificada)
STOPWORDS_ES = {
    "de", "la", "que", "el", "en", "y", "a", "los", "se", "del",
    "las", "un", "por", "con", "no", "una", "su", "para", "es",
    "al", "lo", "como", "más", "o", "pero", "sus", "le", "ha",
    "me", "si", "sin", "sobre", "este", "ya", "entre", "cuando",
    "todo", "esta", "ser", "son", "dos", "también", "fue", "había",
    "era", "muy", "años", "hasta", "desde", "está", "mi", "porque",
    "qué", "sólo", "han", "yo", "hay", "vez", "puede", "todos",
    "así", "nos", "ni", "parte", "tiene", "él", "uno", "donde",
    "bien", "tiempo", "mismo", "ese", "ahora", "cada", "e", "vida",
    "otro", "después", "te", "otros", "aunque", "esa", "eso", "hace",
    "otra", "gobierno", "tan", "durante", "siempre", "día", "grande",
    "cosas", "política", "pequeño", "embargo", "puesto", "tipo", "ahí",
    "aquel", "sea", "según", "menos", "mundo", "año", "dijo", "sido",
    "hacer", "mientras", "cual", "cómo", "cuál", "quién", "quien",
    "dónde", "donde", "cuándo", "cuanto", "cuánto", "tanto", "tal",
    "tan", "así", "solo", "sólo", "aún", "aun", "quizá", "quizás",
}

def extraer_palabras_clave(texto):
    """Extrae palabras clave filtrando stopwords y palabras cortas."""
    # Limpieza básica: minúsculas, quitar signos de puntuación comunes
    texto = texto.lower()
    for signo in "¿?¡!.,;:()[]{}\"''":
        texto = texto.replace(signo, " ")
    
    palabras = texto.split()
    # Filtrar stopwords y palabras de menos de 3 caracteres
    claves = [p for p in palabras if p not in STOPWORDS_ES and len(p) >= 3]
    return claves

# Prueba
test_query = "¿Cuál es la capital de Francia?"
print(f"Consulta: {test_query}")
print(f"Palabras clave extraídas: {extraer_palabras_clave(test_query)}")

In [ ]:
def simple_retrieval(query, documents, top_k=3):
    """
    Recupera los documentos más relevantes usando coincidencia de palabras clave.
    
    Args:
        query (str): La pregunta del usuario.
        documents (list): Lista de documentos.
        top_k (int): Número máximo de documentos a devolver.
    
    Returns:
        list: Documentos relevantes (puede estar vacía si no hay coincidencias).
    """
    palabras_clave = extraer_palabras_clave(query)
    
    if not palabras_clave:
        return []
    
    # Puntuar cada documento por número de coincidencias
    documentos_puntuados = []
    for doc in documents:
        doc_lower = doc.lower()
        coincidencias = sum(1 for palabra in palabras_clave if palabra in doc_lower)
        if coincidencias > 0:
            documentos_puntuados.append((coincidencias, doc))
    
    # Ordenar por número de coincidencias (descendente)
    documentos_puntuados.sort(key=lambda x: x[0], reverse=True)
    
    # Devolver los top_k documentos
    return [doc for _, doc in documentos_puntuados[:top_k]]

# Prueba de la función de recuperación
test_query = "¿Qué es la inteligencia artificial?"
relevant_docs = simple_retrieval(test_query, documents)

print(f"🔍 Prueba de recuperación para: '{test_query}'")
print(f"Documentos relevantes encontrados: {len(relevant_docs)}")
for i, doc in enumerate(relevant_docs, 1):
    print(f"{i}. {doc}")

# Función de generación de respuesta

La función `generate_response` toma la pregunta y el contexto recuperado, y le pide al LLM que responda **basándose únicamente en el contexto**.

In [ ]:
def generate_response(client, query, context, modelo=MODELO):
    """Genera una respuesta basada en el contexto recuperado."""
    system_prompt = (
        "Eres un asistente experto que responde preguntas basándose ÚNICAMENTE "
        "en el contexto proporcionado. Si la información no está en el contexto, "
        "responde claramente: 'No tengo información sobre eso en mi base de conocimiento.' "
        "No inventes ni agregues información externa."
    )
    
    user_prompt = f"""Contexto:
{context}

Pregunta: {query}

Respuesta:"""
    
    response = client.chat.completions.create(
        model=modelo,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0.1,  # Baja temperatura para fidelidad al contexto
        max_tokens=500,
        extra_headers=DEFAULT_HEADERS,
    )
    
    return response

print("✅ Función de generación definida")

# Función RAG completa (evita repetir código)

En lugar de repetir el mismo código en cada ejemplo, definimos una función `rag_query()` que hace todo el flujo: retrieval → generation → muestra resultados.

In [ ]:
def rag_query(query, documents, client, modelo=MODELO, verbose=True):
    """Ejecuta el flujo RAG completo: recuperación + generación."""
    if verbose:
        print(f"❓ Pregunta: {query}")
        print("\n" + "="*50)
    
    # Paso 1: Recuperación
    relevant_docs = simple_retrieval(query, documents)
    
    if verbose:
        print(f"📋 Documentos relevantes ({len(relevant_docs)}):")
        for i, doc in enumerate(relevant_docs, 1):
            print(f"{i}. {doc}")
    
    # Paso 2: Generación
    if relevant_docs:
        context = "\n".join(relevant_docs)
        response = generate_response(client, query, context, modelo=modelo)
        
        if verbose:
            print(f"\n🤖 Respuesta:")
            print(response.choices[0].message.content)
            print(f"\n📊 Tokens usados: {response.usage.total_tokens}")
    else:
        if verbose:
            print("❌ No se encontraron documentos relevantes.")
            print("🤖 Respuesta: No tengo información sobre eso en mi base de conocimiento.")
    
    if verbose:
        print("\n" + "="*50 + "\n")
    
    return relevant_docs

print("✅ Función RAG completa definida")

# Sistema RAG - Ejemplo 1: Pregunta con contexto directo

In [ ]:
rag_query("¿Qué es la inteligencia artificial?", documents, client)

# Sistema RAG - Ejemplo 2: Pregunta sobre RAG

In [ ]:
rag_query("¿Cómo funciona RAG?", documents, client)

# Sistema RAG - Ejemplo 3: Pregunta sin contexto relevante

Este es el caso más interesante: la pregunta no tiene relación con los documentos. Con el filtro de stopwords corregido, ahora `simple_retrieval` **no encuentra coincidencias**, y el LLM responde correctamente que no tiene información.

In [ ]:
rag_query("¿Cuál es la capital de Francia?", documents, client)

# Sistema RAG - Ejemplo 4: Pregunta sobre LangChain

In [ ]:
rag_query("¿Qué es LangChain y para qué sirve?", documents, client)

# Implementación con Streamlit

El título de este notebook promete una implementación con Streamlit. Como Streamlit **no se ejecuta dentro de un notebook**, el código debe ir en un archivo `.py` aparte.

## ¿Cómo funciona Streamlit?
- Es un framework de Python para crear apps web interactivas con muy poco código.
- Se ejecuta desde la terminal con: `streamlit run app.py`
- No requiere conocimientos de HTML, CSS o JavaScript.

## Archivo `app.py` (guárdalo por separado)

```python
# app.py - Aplicación RAG con Streamlit
import os
import streamlit as st
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

# Configuración de la página
st.set_page_config(
    page_title="RAG Básico",
    page_icon="📚",
    layout="centered"
)

# Título y descripción
st.title("📚 RAG Básico con OpenRouter")
st.markdown("Sistema de Recuperación-Aumentada por Generación (RAG) con búsqueda por palabras clave.")

# Inicialización del cliente
@st.cache_resource
def get_client():
    return OpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=os.environ.get("OPENROUTER_API_KEY")
    )

client = get_client()
MODELO = "minimax/minimax-m3:free"

# Base de documentos (en un caso real vendría de un archivo)
documents = [
    "La inteligencia artificial es una rama de la informática que busca crear máquinas capaces de realizar tareas que requieren inteligencia humana.",
    "Los modelos de lenguaje grande (LLM) son sistemas de IA entrenados en enormes cantidades de texto para generar y comprender lenguaje natural.",
    "RAG (Retrieval-Augmented Generation) combina la búsqueda de información relevante con la generación de texto para producir respuestas más precisas.",
    "LangChain es un framework que facilita el desarrollo de aplicaciones con modelos de lenguaje.",
    "El prompt engineering es la práctica de diseñar instrucciones efectivas para obtener los mejores resultados de los modelos de IA."
]

STOPWORDS_ES = {"de", "la", "que", "el", "en", "y", "a", "los", "se", "del", "las", "un", "por", "con", "no", "una", "su", "para", "es", "al", "lo", "como", "más", "o", "pero", "sus", "le", "ha", "me", "si", "sin", "sobre", "este", "ya", "entre", "cuando", "todo", "esta", "ser", "son", "dos", "también", "fue", "había", "era", "muy", "años", "hasta", "desde", "está", "mi", "porque", "qué", "sólo", "han", "yo", "hay", "vez", "puede", "todos", "así", "nos", "ni", "parte", "tiene", "él", "uno", "donde", "bien", "tiempo", "mismo", "ese", "ahora", "cada", "e", "vida", "otro", "después", "te", "otros", "aunque", "esa", "eso", "hace", "otra", "gobierno", "tan", "durante", "siempre", "día", "grande", "cosas", "política", "pequeño", "embargo", "puesto", "tipo", "ahí", "aquel", "sea", "según", "menos", "mundo", "año", "dijo", "sido", "hacer", "mientras", "cual", "cómo", "cuál", "quién", "quien", "dónde", "cuándo", "cuanto", "cuánto", "tanto", "tal", "solo", "aún", "aun"}

def extraer_palabras_clave(texto):
    texto = texto.lower()
    for signo in "¿?¡!.,;:()[]{}\"''":
        texto = texto.replace(signo, " ")
    return [p for p in texto.split() if p not in STOPWORDS_ES and len(p) >= 3]

def simple_retrieval(query, documents, top_k=3):
    palabras_clave = extraer_palabras_clave(query)
    if not palabras_clave:
        return []
    docs_puntuados = []
    for doc in documents:
        coincidencias = sum(1 for p in palabras_clave if p in doc.lower())
        if coincidencias > 0:
            docs_puntuados.append((coincidencias, doc))
    docs_puntuados.sort(key=lambda x: x[0], reverse=True)
    return [doc for _, doc in docs_puntuados[:top_k]]

def generate_response(query, context):
    system_prompt = "Eres un asistente que responde basándose ÚNICAMENTE en el contexto. Si no está la información, dilo claramente."
    user_prompt = f"Contexto:\n{context}\n\nPregunta: {query}\n\nRespuesta:"
    response = client.chat.completions.create(
        model=MODELO,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0.1,
        max_tokens=500,
    )
    return response.choices[0].message.content

# Interfaz de usuario
query = st.text_input("🔍 Escribe tu pregunta:", placeholder="Ej: ¿Qué es la inteligencia artificial?")

if st.button("Buscar respuesta") and query:
    with st.spinner("Recuperando información..."):
        relevant_docs = simple_retrieval(query, documents)
    
    if relevant_docs:
        st.success(f"✅ Se encontraron {len(relevant_docs)} documentos relevantes")
        with st.expander("📋 Ver documentos recuperados"):
            for i, doc in enumerate(relevant_docs, 1):
                st.markdown(f"**{i}.** {doc}")
        
        with st.spinner("Generando respuesta..."):
            context = "\n".join(relevant_docs)
            respuesta = generate_response(query, context)
        
        st.markdown("### 🤖 Respuesta")
        st.info(respuesta)
    else:
        st.warning("⚠️ No se encontraron documentos relevantes para tu pregunta.")
```

## ¿Cómo ejecutarlo?

1. Guarda el código anterior en un archivo llamado `app.py`.
2. Asegúrate de tener tu clave en un archivo `.env`:
   ```
   OPENROUTER_API_KEY=sk-or-v1-...
   ```
3. Abre una terminal y ejecuta:
   ```bash
   streamlit run app.py
   ```
4. Se abrirá una pestaña en tu navegador con la interfaz interactiva.

## Ejercicios Prácticos

### Ejercicio 1: Ampliar la base de documentos
Agrega 5 documentos más sobre temas que te interesen (historia, ciencia, deportes) y prueba el sistema con preguntas relacionadas.

### Ejercicio 2: Mejorar el retrieval
Investiga sobre **TF-IDF** o **BM25** e implementa una versión mejorada de `simple_retrieval` que pese las palabras por su importancia.

### Ejercicio 3: Añadir citas
Modifica `generate_response` para que el modelo incluya la fuente de cada afirmación (ej. `[Documento 2]`).

### Ejercicio 4: Streamlit avanzado
Añade a la app de Streamlit:
- Un `sidebar` con la lista de documentos cargados.
- Un botón para limpiar la conversación.
- Soporte para cargar documentos desde un archivo `.txt`.

## Conceptos Clave

1. **RAG** combina recuperación y generación para dar respuestas fundamentadas.
2. **Retrieval** simple por palabras clave (con filtro de stopwords) es un buen punto de partida didáctico.
3. **Generación** con `temperature` baja para fidelidad al contexto.
4. **Streamlit** permite crear interfaces web interactivas con muy poco código.
5. **En producción** se usan bases vectoriales y embeddings para retrieval semántico.

## Próximos Pasos

En el siguiente notebook exploraremos **embeddings y bases vectoriales** para hacer retrieval semántico en lugar de por palabras clave.